Code Refractor to optimize messy code and test case generator. 

In [ ]:
!pip install gradio transformers torch accelerate openai bitsandbytes sentencepiece dotenv

In [ ]:
import json
import os
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
# =========================
# 🔹 OpenRouter Setup
# =========================


#connecting to openrouter
load_dotenv(override=True)
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')

# Check the key

if not OPENROUTER_API_KEY:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not OPENROUTER_API_KEY.startswith("sk-or-v1"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif OPENROUTER_API_KEY.strip() != OPENROUTER_API_KEY:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

    
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

In [ ]:
# =========================
# 🔹 Hugging Face Setup
# =========================
HG_TOKEN = os.getenv("HG-TOKEN")

if HG_TOKEN:
    if HG_TOKEN.startswith("hf_") and HG_TOKEN.strip() == HG_TOKEN:
        print("Hugging Face token was found.")
    else:
        print("Hugging Face token format is not correct.")
else:
    print("No Hugging Face token found. Some HF models may fail.")


HF_MODELS = {
    "Phi-3 Mini": "microsoft/Phi-3-mini-4k-instruct",
    "TinyLlama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
}


In [ ]:
loaded_models = {}

def load_hf_model(model_key):
    if model_key in loaded_models:
        return loaded_models[model_key]

    model_name = HF_MODELS[model_key]
    print(f"Loading Hugging Face model: {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        token=HG_TOKEN
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        token=HG_TOKEN,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    loaded_models[model_key] = (tokenizer, model)
    return tokenizer, model

In [ ]:
#building the refractor prompt
def build_refactor_prompt(code):
    return f"""
You are a senior software engineer.

Refactor the following code to:
- Improve readability
- Add type hints (if Python)
- Optimize logic if possible
- Follow best practices
- Keep functionality identical

Return ONLY valid JSON in this format:

{{
  "refactored_code": "...",
  "improvements": ["improvement 1", "improvement 2"]
}}

Code:
{code}
"""

In [ ]:
#building the test case generator prompt
def build_test_prompt(code):
    return f"""
You are a senior software engineer.

Generate pytest unit tests for the following code.
Include:
- Normal cases
- Edge cases
- Failure cases if applicable

Return ONLY valid JSON in this format:

{{
  "unit_tests": "pytest code here"
}}

Code:
{code}
"""

In [ ]:
def generate_with_openrouter(prompt, model_name):
    if not openrouter_client:
        raise ValueError("OpenRouter API key not configured.")

    response = openrouter_client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
    )
    return response.choices[0].message.content.strip()

def generate_with_huggingface(prompt, model_key):
    tokenizer, model = load_hf_model(model_key)

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    output = model.generate(
        **inputs,
        max_new_tokens=1500,
        temperature=0.5,
        do_sample=True
    )

    generated_text = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return generated_text.strip()


In [ ]:
def validate_json(text):
    try:
        return json.loads(text), None
    except Exception as e:
        return None, str(e)

In [ ]:
def run_assistant(task_type, code, model_source, model_choice):

    if task_type == "Refactor":
        prompt = build_refactor_prompt(code)
    else:
        prompt = build_test_prompt(code)

    try:
        if model_source == "OpenRouter":
            raw_output = generate_with_openrouter(prompt, model_choice)
        else:
            raw_output = generate_with_huggingface(prompt, model_choice)

        data, error = validate_json(raw_output)

        if error:
            return f"JSON Error: {error}", raw_output

        return "Success", json.dumps(data, indent=2)

    except Exception as e:
        return f"Error: {str(e)}", None

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## Code Refactoring & Test Generator Assistant")

    task_type = gr.Radio(
        ["Refactor", "Generate Tests"],
        value="Refactor",
        label="Task Type"
    )

    code_input = gr.Textbox(
        label="Paste Code Here",
        lines=15
    )

    model_source = gr.Radio(
        ["OpenRouter", "Hugging Face"],
        value="Hugging Face",
        label="Model Source"
    )

    model_choice = gr.Dropdown(
        choices=["Phi-3 Mini", "TinyLlama", "openai/gpt-4o-mini"],
        value="Phi-3 Mini",
        label="Model"
    )

    status = gr.Textbox(label="Status")
    output_box = gr.Textbox(label="Output", lines=20)

    run_btn = gr.Button("Run")

    run_btn.click(
        run_assistant,
        inputs=[task_type, code_input, model_source, model_choice],
        outputs=[status, output_box]
    )

demo.launch()